In [1]:
# train_model.py
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
# merge_dataset.py
import pandas as pd
import glob

In [16]:
# train_model.py

import pandas as pd
import glob
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# =========================
# Load dataset files
# =========================
files = sorted(glob.glob("dataset_*.csv"))

if len(files) == 0:
    raise ValueError("No dataset files found")

print(f"[+] Found {len(files)} dataset files")

# =========================
# Load and clean all data
# =========================
dfs = []
for f in files:
    df = pd.read_csv(f)
    df.dropna(inplace=True)
    df["source_file"] = f  # track origin
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)

print(f"[+] Total rows: {len(full_df)}")
print("\n[+] Overall label distribution:")
print(full_df["label"].value_counts())

# =========================
# Split EACH CLASS separately
# =========================
train_parts = []
test_parts = []

for label in full_df["label"].unique():
    class_df = full_df[full_df["label"] == label].sample(frac=1, random_state=42)

    split_idx = int(len(class_df) * 0.8)

    train_parts.append(class_df.iloc[:split_idx])
    test_parts.append(class_df.iloc[split_idx:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

# Shuffle
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n[+] Train distribution:")
print(train_df["label"].value_counts())

print("\n[+] Test distribution:")
print(test_df["label"].value_counts())

# =========================
# Feature selection
# =========================
FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio"
]

X_train = train_df[FEATURES].values
y_train = (train_df["label"] == "malicious").astype(int).values

X_test = test_df[FEATURES].values
y_test = (test_df["label"] == "malicious").astype(int).values

# =========================
# Train model
# =========================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\n[+] Training model...")
model.fit(X_train, y_train)

# =========================
# Evaluation
# =========================
y_pred = model.predict(X_test)

print("\n[+] Classification Report:")
print(classification_report(y_test, y_pred))

print("\n[+] Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\n[+] Key Metrics:")
print(f"False Positives (benign -> malicious): {fp}")
print(f"False Negatives (missed attacks): {fn}")
print(f"True Positives: {tp}")
print(f"True Negatives: {tn}")

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\nFalse Positive Rate (FPR): {fpr:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")

# =========================
# Feature importance
# =========================
print("\n[+] Feature Importances:")
for name, importance in zip(FEATURES, model.feature_importances_):
    print(f"{name:20s} -> {importance:.4f}")

# =========================
# Save model
# =========================
joblib.dump(model, "rf_model.pkl")
joblib.dump(FEATURES, "features.pkl")

print("\n[+] Model and features saved")

[+] Found 3 dataset files
[+] Total rows: 60243

[+] Overall label distribution:
label
benign       30126
malicious    30117
Name: count, dtype: int64

[+] Train distribution:
label
benign       24100
malicious    24093
Name: count, dtype: int64

[+] Test distribution:
label
benign       6026
malicious    6024
Name: count, dtype: int64

[+] Training model...

[+] Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6026
           1       1.00      1.00      1.00      6024

    accuracy                           1.00     12050
   macro avg       1.00      1.00      1.00     12050
weighted avg       1.00      1.00      1.00     12050


[+] Confusion Matrix:
[[6026    0]
 [   0 6024]]

[+] Key Metrics:
False Positives (benign -> malicious): 0
False Negatives (missed attacks): 0
True Positives: 6024
True Negatives: 6026

False Positive Rate (FPR): 0.0000
False Negative Rate (FNR): 0.0000

[+] Feature Importances:
de

In [15]:
# Merge all collected CSVs
files = glob.glob("dataset_*.csv")  # if you saved multiple sessions

if len(files) < 2:
    raise ValueError("You need at least 2 dataset files for proper train/test split")

print(f"[+] Found {len(files)} dataset files")

# Use some files for training, others for testing
train_files = files[:-1]
test_files = files[-1:]

print(f"[+] Training on: {train_files}")
print(f"[+] Testing on: {test_files}")

train_df = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
test_df  = pd.concat([pd.read_csv(f) for f in test_files], ignore_index=True)

print("\n[+] Train label distribution:")
print(train_df["label"].value_counts())

print("\n[+] Test label distribution:")
print(test_df["label"].value_counts())

# Drop NaNs

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio"
]

X_train = train_df[FEATURES].values
y_train = (train_df["label"] == "malicious").astype(int).values

X_test = test_df[FEATURES].values
y_test = (test_df["label"] == "malicious").astype(int).values
#df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

model = RandomForestClassifier(
n_estimators=200,
max_depth=10,
min_samples_split=10,
class_weight="balanced",
random_state=42,
n_jobs=-1
)

print("\n[+] Training model...")
model.fit(X_train, y_train)
# Check balance
#print(test_df["label"].value_counts())

y_pred = model.predict(X_test)

print("\n[+] Classification Report:")
print(classification_report(y_test, y_pred))

print("\n[+] Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =========================

# Feature importance (debugging)

# =========================

print("\n[+] Feature Importances:")
for name, importance in zip(FEATURES, model.feature_importances_):
    print(f"{name:20s} -> {importance:.4f}")

# =========================

# Save model

# =========================

joblib.dump(model, "rf_model.pkl")
joblib.dump(FEATURES, "features.pkl")

print("\n[+] Model and features saved")

[+] Found 3 dataset files
[+] Training on: ['dataset_benign.csv', 'dataset_final.csv']
[+] Testing on: ['dataset_malicious.csv']

[+] Train label distribution:
label
benign       30126
malicious    20078
Name: count, dtype: int64

[+] Test label distribution:
label
malicious    10039
Name: count, dtype: int64

[+] Training model...

[+] Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     10039

    accuracy                           1.00     10039
   macro avg       1.00      1.00      1.00     10039
weighted avg       1.00      1.00      1.00     10039


[+] Confusion Matrix:
[[10039]]

[+] Feature Importances:
dest_port            -> 0.0000
window_duration      -> 0.0307
fwd_packet_rate      -> 0.1005
fwd_byte_rate        -> 0.0712
pkt_len_mean         -> 0.0976
pkt_len_std          -> 0.1042
pkt_len_min          -> 0.0239
pkt_len_max          -> 0.3845
syn_ratio            -> 0.0743
fin_ratio            -> 0.0

c:\Users\izzyd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [3]:
# Merge all collected CSVs
files = glob.glob("dataset_*.csv")  # if you saved multiple sessions
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Check balance
print(df["label"].value_counts())

# Check for NaNs
print(df.isnull().sum())

# Drop any bad rows
df.dropna(inplace=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv("dataset_final.csv", index=False)
print(f"[+] Final dataset: {len(df)} rows")

label
benign       10042
malicious    10039
Name: count, dtype: int64
dest_port          0
window_duration    0
fwd_packet_rate    0
fwd_byte_rate      0
pkt_len_mean       0
pkt_len_std        0
pkt_len_min        0
pkt_len_max        0
syn_ratio          0
fin_ratio          0
ack_ratio          0
label              0
dtype: int64
[+] Final dataset: 20081 rows


In [4]:
# Check for NaNs
print(df.isnull().sum())

# Drop any bad rows
df.dropna(inplace=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

dest_port          0
window_duration    0
fwd_packet_rate    0
fwd_byte_rate      0
pkt_len_mean       0
pkt_len_std        0
pkt_len_min        0
pkt_len_max        0
syn_ratio          0
fin_ratio          0
ack_ratio          0
label              0
dtype: int64


In [5]:
df.to_csv("dataset_final.csv", index=False)

print(f"[+] Final dataset: {len(df)} rows")

[+] Final dataset: 20081 rows


In [6]:
df = pd.read_csv("dataset_final.csv")

FEATURES = [
    "dest_port", "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "pkt_len_mean", "pkt_len_std",
    "pkt_len_min", "pkt_len_max",
    "syn_ratio", "fin_ratio", "ack_ratio"
]

In [7]:
print("=== Class Distribution ===")
print(df["label"].value_counts())
print(f"\nRatio: {df['label'].value_counts()['benign'] / df['label'].value_counts()['malicious']:.2f}x more benign than malicious")

print("\n=== Feature Stats by Class ===")
for col in df.columns[:-1]:
    benign_mean    = df[df["label"]=="benign"][col].mean()
    malicious_mean = df[df["label"]=="malicious"][col].mean()
    print(f"{col:20s}  benign={benign_mean:10.4f}  malicious={malicious_mean:10.4f}")

=== Class Distribution ===
label
benign       10042
malicious    10039
Name: count, dtype: int64

Ratio: 1.00x more benign than malicious

=== Feature Stats by Class ===
dest_port             benign=   80.0000  malicious=  552.5098
window_duration       benign=    1.6760  malicious=    4.5725
fwd_packet_rate       benign=   82.7762  malicious= 1035.1560
fwd_byte_rate         benign= 6642.9817  malicious=69056.4301
pkt_len_mean          benign=  201.5109  malicious=  168.6362
pkt_len_std           benign=  291.0675  malicious=  190.1337
pkt_len_min           benign=   66.0185  malicious=   62.7172
pkt_len_max           benign= 1068.1589  malicious=  716.7041
syn_ratio             benign=    0.1568  malicious=    0.4065
fin_ratio             benign=    0.1849  malicious=    0.1289
ack_ratio             benign=    0.9196  malicious=    0.7284


In [9]:
X = df[FEATURES].values
y = (df["label"] == "malicious").astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2009
           1       1.00      1.00      1.00      2008

    accuracy                           1.00      4017
   macro avg       1.00      1.00      1.00      4017
weighted avg       1.00      1.00      1.00      4017



In [11]:
# Save model and feature list
joblib.dump(model, "rf_model.pkl")
joblib.dump(FEATURES, "features.pkl")
print("[+] Model and features saved")

[+] Model and features saved


In [12]:
importance_df = pd.DataFrame({
    "Feature":    FEATURES,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("=== Feature Importance ===")
print(importance_df.to_string())

# Probability distribution — are predictions clustered near 0.5?
probs = model.predict_proba(X)[:, 1]
print(f"\n=== Probability Distribution ===")
print(f"Mean prob (benign windows):    {probs[y==0].mean():.4f}")
print(f"Mean prob (malicious windows): {probs[y==1].mean():.4f}")
print(f"Benign windows predicted >0.7: {(probs[y==0] > 0.7).sum()} / {(y==0).sum()}")

=== Feature Importance ===
            Feature  Importance
7       pkt_len_max    0.413822
5       pkt_len_std    0.095334
2   fwd_packet_rate    0.092454
3     fwd_byte_rate    0.077980
10        ack_ratio    0.071038
4      pkt_len_mean    0.069128
8         syn_ratio    0.060672
9         fin_ratio    0.053827
1   window_duration    0.037614
6       pkt_len_min    0.028131
0         dest_port    0.000000

=== Probability Distribution ===
Mean prob (benign windows):    0.0011
Mean prob (malicious windows): 0.9990
Benign windows predicted >0.7: 0 / 10042


In [13]:
# After fitting the model, check false positive rate specifically
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))

# False positive analysis — benign windows predicted as malicious
fp_mask = (y_test == 0) & (y_pred == 1)
print(f"\nFalse positives: {fp_mask.sum()} / {(y_test==0).sum()} benign windows")
print("\nFalse positive feature averages:")
fp_features = pd.DataFrame(X_test[fp_mask], columns=FEATURES)
print(fp_features.mean().to_string())

# What do these false positives look like?
# This tells you exactly which benign traffic looks malicious to the model

              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00      2009
   Malicious       1.00      1.00      1.00      2008

    accuracy                           1.00      4017
   macro avg       1.00      1.00      1.00      4017
weighted avg       1.00      1.00      1.00      4017


False positives: 0 / 2009 benign windows

False positive feature averages:
dest_port         NaN
window_duration   NaN
fwd_packet_rate   NaN
fwd_byte_rate     NaN
pkt_len_mean      NaN
pkt_len_std       NaN
pkt_len_min       NaN
pkt_len_max       NaN
syn_ratio         NaN
fin_ratio         NaN
ack_ratio         NaN


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Test with increasing tree depths to see where overfitting starts
print("=== Depth vs Accuracy (overfitting check) ===")
for depth in [3, 5, 10, 20, None]:
    rf_test = RandomForestClassifier(
        n_estimators=50, 
        max_depth=depth, 
        random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(rf_test, X, y, cv=cv, scoring="f1")
    print(f"  max_depth={str(depth):>5s}  CV f1={scores.mean():.4f} (+/-{scores.std():.4f})")

# Check if training accuracy vs test accuracy diverge — 
# large gap = overfitting
model.fit(X_train, y_train)
train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test, y_test)
print(f"\nTrain accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Gap:            {train_acc - test_acc:.4f}  (>0.02 suggests overfitting)")

=== Depth vs Accuracy (overfitting check) ===


NameError: name 'RandomForestClassifier' is not defined